In [1]:
#-----Graph Retrieval-Augmented Generation (GraphRAG)---If someone asks, “Why was this treatment given?”, can we retrieve the 
# connected evidence before asking an LLM to formulate an answer?
"""
GraphRAG combines graph-based retrieval with a language model. Instead of searching only for similar text, the system follows 
relationships between clinical events and supporting documents.

For example:
EGFR Result → Oncology Decision → Treatment Start
Each relationship should point to relevant evidence, such as a molecular report or a dated oncology note.

What happens next?---
An LLM can use this retrieved evidence to draft an answer such as:
The available records show that osimertinib was prescribed after an EGFR L858R result and a documented oncology treatment plan. 
The supporting records are the molecular report, oncology note, and prescription.

The answer is deliberately cautious. The retrieved records establish a documented sequence, but the system should not claim the 
exact clinical rationale unless the oncology note actually states it.

Expected result of below program: The system retrieves the treatment record, the oncology decision, and the EGFR result, 
including their dates and source identifiers.
"""
# A small oncology knowledge graph. # Each node stores its source evidence.
graph = {
    "EGFR_Result": {
        "type": "Biomarker",
        "value": "EGFR L858R detected",
        "source": "Molecular_Report_12",
        "date": "2026-01-10",
        "links": ["Oncology_Decision"]
    },
    "Oncology_Decision": {
        "type": "Clinical_Decision",
        "value": "Oncologist documented targeted treatment plan",
        "source": "Oncology_Note_15",
        "date": "2026-01-18",
        "links": ["EGFR_Result", "Treatment_Start"]
    },
    "Treatment_Start": {
        "type": "Treatment",
        "value": "Osimertinib prescribed",
        "source": "Prescription_22",
        "date": "2026-01-20",
        "links": ["Oncology_Decision"]
    }
}
def retrieve_treatment_evidence(treatment_node):
    if treatment_node not in graph:
        return []
    evidence = []
    visited = set()
    queue = [treatment_node]
    while queue:
        node = queue.pop(0)
        if node in visited:
            continue
        visited.add(node)
        record = graph[node]
        evidence.append({
            "node": node,
            "value": record["value"],
            "source": record["source"],
            "date": record["date"]
        })
        for neighbor in record["links"]:
            if neighbor not in visited:
                queue.append(neighbor)
    return evidence

evidence = retrieve_treatment_evidence("Treatment_Start")
print("Retrieved evidence for treatment:")
for item in evidence:
    print(
        item["date"],
        "|", item["value"],
        "| Source:", item["source"]
    )

Retrieved evidence for treatment:
2026-01-20 | Osimertinib prescribed | Source: Prescription_22
2026-01-18 | Oncologist documented targeted treatment plan | Source: Oncology_Note_15
2026-01-10 | EGFR L858R detected | Source: Molecular_Report_12
